# Week 1 — Modern GPU Microarchitecture & the Memory Hierarchy

> *"The first rule of high-performance computing: know your machine."*

This notebook lays the **hardware-level foundation** on which every subsequent week rests. We re-derive the equations governing GPU throughput, measure the bandwidth of every tier of the memory hierarchy, and produce a roofline plot that classifies every kernel of a real model as memory-bound or compute-bound.

## Learning objectives

1. Understand the evolution from **Ampere (A100)** → **Hopper (H100)** → **Blackwell (B200)** at the level of Streaming Multiprocessors (SMs) and Tensor Core generations.
2. Derive the **Roofline model** from first principles and use it as a diagnostic tool.
3. Quantify the **precision lattice** — FP32, TF32, BF16, FP16, FP8 (E4M3, E5M2), FP4 — in terms of dynamic range, mantissa precision, and Tensor Core throughput multiplier.
4. Measure GPU memory consumption with `torch.cuda.memory_allocated` and produce a per-layer footprint report.


## 1. The GPU as a Memory Hierarchy

A modern GPU is best thought of as a **bandwidth-stratified memory pyramid**:

| Tier             | Capacity (H100)  | Bandwidth (peak) | Latency        |
|------------------|------------------|------------------|----------------|
| Registers        | 256 KiB / SM     | ~10 TB/s         | ~1 cycle       |
| Shared memory / L1 | 228 KiB / SM   | ~20 TB/s         | ~30 cycles     |
| L2 cache         | 50 MiB           | ~5.5 TB/s        | ~200 cycles    |
| HBM3             | 80 GiB           | ~3.35 TB/s       | ~600 cycles    |
| NVLink 4         | (off-chip)       | 0.9 TB/s         | ~µs            |
| PCIe Gen5        | (off-chip)       | 64 GB/s          | ~µs            |

The performance of any GPU kernel is determined by **how well its access pattern matches the bandwidth of the tier it touches**. This observation gives rise to the roofline model.


## 2. The Roofline Model

For a kernel with **arithmetic intensity**

$$
I = \frac{\text{FLOPs}}{\text{Bytes moved from HBM}}
$$

the attainable performance is

$$
P_{\text{attainable}}(I) \;=\; \min\bigl(P_{\text{peak}},\; B_{\text{peak}}\cdot I\bigr).
$$

The transition between the two regimes occurs at the **ridge point** $I^{*} = P_{\text{peak}} / B_{\text{peak}}$ (the "machine balance"). For H100 BF16:

$$
I^{*}_{\text{H100, BF16}} \;=\; \frac{989\ \text{TFLOPS}}{3.35\ \text{TB/s}} \;\approx\; 295\ \text{FLOPs / byte}.
$$

Any kernel with $I < 295$ is HBM-bound on H100; the only way to speed it up is to **reduce byte traffic** (fusion, recomputation, lower precision), not to add more compute.


In [ ]:
# %% Measure peak HBM bandwidth empirically with a streaming copy
import torch

assert torch.cuda.is_available(), "This notebook requires a CUDA device."

def measure_hbm_bandwidth(size_mb: int = 1024, iters: int = 50) -> float:
    """Return measured HBM bandwidth in GB/s via a torch.empty_like copy."""
    n_bytes = size_mb * 1024 * 1024
    src = torch.empty(n_bytes // 4, dtype=torch.float32, device='cuda')
    dst = torch.empty_like(src)

    # Warm-up
    for _ in range(5):
        dst.copy_(src)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        dst.copy_(src)
    end.record()
    torch.cuda.synchronize()

    elapsed_s = start.elapsed_time(end) * 1e-3 / iters
    # Each copy reads n_bytes and writes n_bytes -> 2 * n_bytes total
    return (2 * n_bytes) / elapsed_s / 1e9

bw = measure_hbm_bandwidth()
print(f"Measured HBM bandwidth: {bw:,.1f} GB/s")


## 3. The Precision Lattice

The trade-off space spanned by modern numeric formats:

| Dtype     | Bits | Exp | Mant | Max value     | Rel. ε       | TC throughput vs FP32 |
|-----------|------|-----|------|---------------|--------------|------------------------|
| FP32      | 32   | 8   | 23   | ~3.4·10³⁸     | 2⁻²⁴ ≈ 6e-8  | 1×                     |
| TF32      | 19   | 8   | 10   | ~3.4·10³⁸     | 2⁻¹¹ ≈ 5e-4  | 8× (A100)              |
| BF16      | 16   | 8   | 7    | ~3.4·10³⁸     | 2⁻⁸  ≈ 4e-3  | 16× (A100), 32× (H100) |
| FP16      | 16   | 5   | 10   | 65,504        | 2⁻¹¹ ≈ 5e-4  | 16× (A100)             |
| FP8 E4M3  | 8    | 4   | 3    | 448           | 2⁻⁴  ≈ 6e-2  | 64× (H100)             |
| FP8 E5M2  | 8    | 5   | 2    | 57,344        | 2⁻³  ≈ 1e-1  | 64× (H100)             |
| FP4 / NF4 | 4    | 2   | 1    | depends       | ~0.25        | 128× (B200)            |

The **mantissa precision** bounds the worst-case relative quantization error by $\varepsilon_{\text{rel}} = 2^{-(m+1)}$ for $m$ mantissa bits. The **exponent width** determines the dynamic range and, crucially, whether gradients underflow during training.


In [ ]:
# %% Per-dtype memory footprint of a 350M-parameter model
from hpcllmforge.memory.precision import (
    FP32, TF32, BF16, FP16, FP8_E4M3, FP8_E5M2, FP4, memory_footprint_bytes,
)

N = 350_000_000  # parameter count

for dtype in [FP32, BF16, FP16, FP8_E4M3, FP4]:
    bytes_ = memory_footprint_bytes(N, dtype)
    print(f"{dtype.name:>10}: {bytes_/1024/1024:>8,.0f} MiB")


## 4. Mixed-Precision Memory Accounting

A typical mixed-precision Adam training run holds **eight copies of the model** in some form:

$$
M_{\text{total}} \;=\; \underbrace{2N}_{\text{fp16 params}} + \underbrace{2N}_{\text{fp16 grads}} + \underbrace{4N + 4N + 4N}_{\text{fp32 master + Adam m + Adam v}} + M_{\text{activations}}
$$

so even before counting activations, **each parameter costs 16 bytes**. For a 1.3B-parameter model that is **20.8 GiB** of state — already a substantial fraction of an A100-80GB.

This equation is the motivation for everything that follows in Weeks 2–4.


In [ ]:
# %% Measure live memory before/after a training step
import torch
import torch.nn as nn
from torch.optim import AdamW

device = 'cuda'
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = nn.TransformerEncoderLayer(d_model=1024, nhead=16, dim_feedforward=4096).to(device)
optim = AdamW(model.parameters(), lr=1e-4)

x = torch.randn(32, 128, 1024, device=device, requires_grad=True)
y = model(x).sum()
y.backward()
optim.step()

print(f"Peak memory : {torch.cuda.max_memory_allocated() / 1e9:.3f} GB")
print(f"Live memory : {torch.cuda.memory_allocated() / 1e9:.3f} GB")


## 5. Exercises

1. Repeat the bandwidth measurement for buffer sizes from 1 MiB to 1 GiB and plot the curve. Identify the L2-resident regime and the HBM-bound regime.
2. Compute the ridge point of your device for FP32, BF16, and FP8 throughput. For each precision, classify a matmul of shape $(8192, 8192) \times (8192, 8192)$ as memory-bound or compute-bound.
3. Run a forward pass of GPT-2 small and produce a per-layer memory trace using `torch.cuda.memory._record_memory_history`.
